In [1]:
import os
from elasticsearch import Elasticsearch
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from extraction_prompts import (
    EXTRACT_RELATION_TRIPLETS_PROMPT,
    REFORMULATE_RELATION_TRIPLET,
    NATURAL_LANGUAGE_PROMPT

)
from utils import ollama_request

NLI_MODEL_NAME = "facebook/bart-large-mnli"
ES_HOST = os.getenv("ES_HOST", "http://localhost:9200")
INDEX_NAME = os.getenv("INDEX_NAME")
FRAGMENT_INDEX_NAME = f"{os.getenv('INDEX_NAME')}_fragments"
SAMPLE_FRAGMENTS_INDEX_NAME = f"{os.getenv('INDEX_NAME')}_fragments_sample"
CHUNK_3_SAMPLE_FRAGMENTS_INDEX_NAME = f"{os.getenv('INDEX_NAME')}_fragments_sample_chunk_3"
es_client = Elasticsearch('http://localhost:9200')

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_relations(
        es_client,
        fragment_index_name,
        reformulate_predicate=False,
        speech_id_start=1,
        speech_id_end=10,
        dataset_path="data/triplets.csv",
        reformulations_path="data/reformulations.json",
        extractor_model_name=os.getenv("EXTRACTOR_MODEL_NAME"),
        reformulate_model_name=os.getenv("REFORMULATE_MODEL_NAME"),
        sample_list=None):
    try:
        final_triplets = pd.read_csv(dataset_path)
    except:
        final_triplets = pd.DataFrame(columns=["speech_id", "fragment_id", "date", "subject", "predicate", "object"])
    if sample_list is None:
        sample_list = list(range(speech_id_start, speech_id_end + 1))

    for speech_id in tqdm(sample_list, f"Processing speeches for model {extractor_model_name}"):
        if int(speech_id) in final_triplets["speech_id"].values:
            print(f"Speech {speech_id} already processed, skipping...")
            continue
        fragments_for_text = es_client.search(
            index=fragment_index_name,
            query={
                "match": {
                    "speech_id": speech_id
                }
            },
            size=10000
        )
        hits = fragments_for_text['hits']['hits']
        if len(hits) == 0:
            continue
        new_triplets = pd.DataFrame(columns=["speech_id", "fragment_id", "date", "subject", "predicate", "object"])
        for fragment_id in range(0, len(hits)):
            fragment = hits[fragment_id]
            fragment_text = fragment["_source"]["text"]
            jsoned_triplets = None
            triplets = ollama_request(
                prompt=EXTRACT_RELATION_TRIPLETS_PROMPT.format(fragment=fragment_text, speech_author=fragment["_source"].get("author", "Unknown")),
                is_stream=False,
                model_name=extractor_model_name
            ).replace("```json", "").replace("```", "").replace("*", "").strip()
            try:
                jsoned_triplets = json.loads(triplets)["triplets"]
            except Exception as e:
                    continue
                # else:
                #     print(f"JSON decoding error for speech_id {speech_id}, fragment_id {fragment_id}, Retrying once with the same prompt...")
            reformulations = {}
            for triplet in jsoned_triplets:
                if triplet is None or not isinstance(triplet, dict):
                    continue
                subject = triplet.get("subject", None)
                predicate = triplet.get("predicate", None)
                object_ = triplet.get("object", None)

                if object_:
                    object_ = str(object_).strip()

                if any(not x for x in [subject, predicate, object_]):
                    continue

                forbidden_entities = {
                    "unknown",
                    "i", "me", "we",
                    "you",
                    "he", "him", "she", "her",
                    "they", "them", "it",
                    "my", "mine", "our", "ours",
                    "your", "yours", "his", "hers",
                    "their", "theirs", "its",
                    "myself", "ourselves",
                    "yourself", "yourselves",
                    "himself", "herself",
                    "themselves", "itself",
                    "this", "that", "these", "those"
                }
                
                if subject.lower() in forbidden_entities or object_.lower() in forbidden_entities:
                    continue

                reformulated_triplet = {}
                if reformulate_predicate:
                    reformulated_predicate = ollama_request(
                            prompt=REFORMULATE_RELATION_TRIPLET.format(subject=subject, predicate=predicate, object=object_),
                            is_stream=False,
                            model_name=reformulate_model_name
                    ).replace("```json", "").replace("```", "").replace("*", "").replace('<', "").replace('>', "").replace('.', "").strip()
                    print(f"Changed {predicate} to: {reformulated_predicate}")
                    reformulated_triplet["predicate"] = reformulated_predicate
                    reformulations[predicate] = reformulated_predicate

                reformulated_triplet = {
                    "subject": reformulated_triplet.get("subject", subject).split("(")[0].strip(),
                    "predicate": reformulated_triplet.get("predicate", predicate),
                    "object": reformulated_triplet.get("object", object_).split("(")[0].strip()
                }
                new_triplets = pd.concat([new_triplets, pd.DataFrame([{
                    "speech_id": speech_id,
                    "fragment_id": fragment["_id"],
                    "date": fragment["_source"]["date"],
                    "subject": reformulated_triplet.get("subject", subject).lower().replace('<', ' ').replace('>', ' ').replace('.', ' ').replace("'", '').strip(),
                    "predicate": reformulated_triplet.get("predicate", predicate).replace('<', ' ').replace('>', ' ').replace('.', ' ').replace("'", '').lower().strip(),
                    "object": reformulated_triplet.get("object", object_).lower().replace('<', ' ').replace('>', ' ').replace('.', ' ').replace("'", '').strip()
                }])], ignore_index=True)
        final_triplets = pd.concat([final_triplets[final_triplets["speech_id"] != speech_id], new_triplets], ignore_index=True)
        final_triplets.to_csv(dataset_path, index=False)
    with open(reformulations_path, "w") as f:
        json.dump(reformulations, f, indent=4)

### Final Run

In [3]:
with open("data/sampled_ids_1000.json", "r") as f:
    sampled_ids = json.load(f)

create_relations(
    es_client=es_client,
    fragment_index_name=SAMPLE_FRAGMENTS_INDEX_NAME,
    dataset_path=f"data/triplets_extracted_1000.csv",
    reformulate_predicate=False,
    speech_id_start=0,
    speech_id_end=10000,
    extractor_model_name="ministral-3:3b",
    sample_list=sampled_ids
)

Processing speeches for model ministral-3:3b:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing speeches for model ministral-3:3b: 100%|██████████| 1000/1000 [3:07:30<00:00, 11.25s/it]  


### Tests

#### Triplet Extraction

In [3]:
MODELS_TO_TEST = {
    # "qwen": "qwen2.5:3b",
    "ministral": "ministral-3:3b",
    # "llama": "llama3.2:latest",
}

with open("data/sampled_ids_1000.json", "r") as f:
    sampled_ids = json.load(f)

In [8]:
# My text fragmentation

for model_name, model_id in MODELS_TO_TEST.items():
    create_relations(
        es_client=es_client,
        fragment_index_name=SAMPLE_FRAGMENTS_INDEX_NAME,
        dataset_path=f"data/triplets_{model_name}_custom_chunk_1000.csv",
        reformulate_predicate=False,
        speech_id_start=0,
        speech_id_end=10000,
        extractor_model_name=model_id,
        sample_list=sampled_ids
    )

Processing speeches for model ministral-3:3b: 100%|██████████| 1000/1000 [1:21:32<00:00,  4.89s/it] 


In [ ]:
# Chunk 3 fragmentation

for model_name, model_id in MODELS_TO_TEST.items():
    create_relations(
        es_client=es_client,
        fragment_index_name=CHUNK_3_SAMPLE_FRAGMENTS_INDEX_NAME,
        dataset_path=f"data/triplets_{model_name}_chunk_3.csv",
        reformulate_predicate=False,
        speech_id_start=0,
        speech_id_end=10000,
        extractor_model_name=model_id,
        sample_list=sampled_ids
    )

Processing speeches for model llama3.2:latest: 100%|██████████| 100/100 [23:36<00:00, 14.16s/it] 


#### Triplet Tests

In [3]:
NLI_TOKENIZER = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
NLI_MODEL = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)

In [4]:
def nli_score(premise, hypothesis):
    inputs = NLI_TOKENIZER(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        logits = NLI_MODEL(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]
    scores = {
        "contradiction": probs[0].item(),
        "neutral": probs[1].item(),
        "entailment": probs[2].item()
    }

    return scores["entailment"]

In [5]:
def evaluate_triplets(df, fragmentation_type):
    fragment_index_name = SAMPLE_FRAGMENTS_INDEX_NAME if fragmentation_type == "custom" else CHUNK_3_SAMPLE_FRAGMENTS_INDEX_NAME
    words_to_replace = ["i", "me", "my"]
    fragments_ids = df["fragment_id"].unique()
    result_faithfulness = []
    result_completeness = []
    for fragment_id in tqdm(fragments_ids, f"Evaluating triplets for {len(fragments_ids)} fragments"):
        source_fragment = es_client.search(index=fragment_index_name, query={"match": {"_id": fragment_id}}, size=100)["hits"]["hits"][0]
        source_fragment_text = source_fragment["_source"]["text"]
        source_fragment_author = source_fragment["_source"].get("author", "Unknown")

        source_fragment_text_list = source_fragment_text.split(" ")
        for i in range(len(source_fragment_text_list)):
            if source_fragment_text_list[i].lower() in words_to_replace:
                source_fragment_text_list[i] = source_fragment_author
        source_fragment_text = " ".join(source_fragment_text_list)

        fragment_triplets = df[df["fragment_id"] == fragment_id]

        triplet_sentences = []
        for idx, r in fragment_triplets.iterrows():
            triplet = " ".join([str(r["subject"]), str(r["predicate"]), str(r["object"])])
            sentence_triplet = ollama_request(
                    prompt=NATURAL_LANGUAGE_PROMPT.format(triplet=triplet),
                    is_stream=False,
                    model_name="ministral-3:3b"
            ).replace("```json", "").replace("```", "").replace("*", "").replace('<', "").replace('>', "").replace('.', "").strip()
            triplet_sentences.append(sentence_triplet)

        joined_triplet_sentences = ". ".join(triplet_sentences)

        faithfulness_scores = []
        for sent in triplet_sentences:
            faithfulness_scores.append(nli_score(source_fragment_text, sent))
        result_faithfulness.append(np.mean(faithfulness_scores))
        result_completeness.append(nli_score(joined_triplet_sentences, source_fragment_text))
    return np.mean(result_faithfulness), np.mean(result_completeness)

In [ ]:
extraction_scores = {
    'ministral_custom_chunk': {'faithfulness': 0.575, 'completeness': 0.0901},
    'ministral_chunk_3': {'faithfulness': 0.5186, 'completeness': 0.0538},
    'qwen_custom_chunk': {'faithfulness': 0.5567, 'completeness': 0.1387},
    'qwen_chunk_3': {'faithfulness': 0.5282, 'completeness': 0.0942},
    'llama_custom_chunk': {'faithfulness': 0.4425, 'completeness': 0.1502},
    'llama_chunk_3': {'faithfulness': 0.444, 'completeness': 0.0766}
}

for key in extraction_scores.keys():
    df = pd.read_csv(f"data/triplets_{key}.csv")
    fragmentation_type = "custom" if "custom" in key else "chunk_3"
    faithfulness, completeness = evaluate_triplets(df, fragmentation_type)
    extraction_scores[key]["faithfulness"] = round(float(faithfulness), 4)
    extraction_scores[key]["completeness"] = round(float(completeness), 4)

Evaluating triplets for 1337 fragments: 100%|██████████| 1337/1337 [40:12<00:00,  1.80s/it] 


#### Prepare Sample Dataset For Reference Solution

In [4]:
sample_df = pd.DataFrame()

for speech_id in sampled_ids:
    fragments_for_text = es_client.search(
        index=INDEX_NAME,
        query={
            "match": {
                "_id": speech_id
            }
        },
        size=10000
    )
    hits = fragments_for_text['hits']['hits'][0]["_source"]
    if sample_df.empty:
        sample_df = pd.DataFrame([hits])
    else:
        sample_df = pd.concat([sample_df, pd.DataFrame([hits])], ignore_index=True)

sample_df = sample_df[["date", "text"]]
sample_df["date"] = pd.to_datetime(sample_df["date"]).dt.strftime("%Y-%m-%d")

In [ ]:
years = [i for i in range(2000, 2026)]

for year in years:
    year_df = sample_df[sample_df["date"].str.startswith(str(year))]
    year_df.to_csv(f"data/notebooklm_dfs/{year}.csv", index=False)